# ChemBreak V11 - Colab Enterprise deployment

This is the primary V11 notebook for Google Cloud Colab Enterprise.

Key differences from the standard Google Colab notebook:

- Uses Colab Enterprise Application Default Credentials (ADC).
- Does not mount Google Drive.
- Uses a Cloud Storage bucket for durable checkpoints.
- Restores previous checkpoints from Cloud Storage after a runtime restart or replacement.
- Mirrors changed output files to Cloud Storage every 60 seconds while a stage is running.
- Keeps the V11 live progress, heartbeat, ETA, repair, concurrent judging, adjudication, and refill logic unchanged.

Start with `RUN_TYPE = "test"`.


## 1. Enterprise configuration

The default bucket name is derived from the Google Cloud project ID. If your organization does not allow bucket creation, replace `GCS_BUCKET` with an existing bucket that you can read and write.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time
import threading
import hashlib

import google.auth
from google.cloud import storage
from google.api_core.exceptions import NotFound, Forbidden

PROJECT_ID = "rs-foundsecft-mghasemi"
REPO_URL = "https://github.com/Jollychuks/ChemBreak.git"
PROJECT_SUBDIR = "ChemBreak_V11_Cloud"

# Change only if your institution requires an existing bucket.
GCS_BUCKET = f"{PROJECT_ID}-chembreak-v11"
GCS_BUCKET_LOCATION = "US"
GCS_PREFIX = "ChemBreak_V11"

# Durable mirror interval while a stage is running.
GCS_SYNC_SECONDS = 60

# Colab Enterprise normally provides ADC when end-user credentials are enabled.
credentials, adc_project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_QUOTA_PROJECT"] = PROJECT_ID

print("ADC detected:", type(credentials).__name__)
print("ADC project:", adc_project)
print("Quota project:", PROJECT_ID)

# Runtime-local repository. It can disappear if the runtime is deleted.
RUNTIME_ROOT = Path.home() / "chembreak_v11_runtime"
REPO_ROOT = RUNTIME_ROOT / "ChemBreak_repo"
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)

if not REPO_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)

PROJECT_DIR = REPO_ROOT / PROJECT_SUBDIR
if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"{PROJECT_SUBDIR} was not found in {REPO_URL}. "
        "Upload the V11 folder to the GitHub repository first."
    )

PIPELINE = PROJECT_DIR / "scripts" / "chembreak_v11_cloud.py"
CONFIG_SOURCE = PROJECT_DIR / "config" / "run_config.json"

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_DIR / "requirements.txt")],
    check=True,
)

pipeline_source = PIPELINE.read_text(encoding="utf-8")
required_markers = [
    'VERSION = "11.0-cloud"',
    'NAMESPACE = "CBV11C"',
    "ThreadPoolExecutor",
    "prejudge_refill_stage",
]
missing = [m for m in required_markers if m not in pipeline_source]
if missing:
    raise RuntimeError(f"Wrong or incomplete V11 pipeline loaded: {missing}")

print("Project directory:", PROJECT_DIR)
print("Pipeline verification: PASSED")


## 2. Durable Cloud Storage checkpointing

Colab Enterprise runtime files are not the durable source of truth. The notebook creates or opens the configured bucket and keeps the V11 output tree mirrored there.


In [ ]:
storage_client = storage.Client(project=PROJECT_ID, credentials=credentials)

try:
    bucket = storage_client.get_bucket(GCS_BUCKET)
    print("Using existing bucket:", GCS_BUCKET)
except NotFound:
    try:
        bucket = storage_client.create_bucket(
            GCS_BUCKET,
            project=PROJECT_ID,
            location=GCS_BUCKET_LOCATION,
        )
        print("Created bucket:", GCS_BUCKET)
    except Forbidden as exc:
        raise RuntimeError(
            "You can access Colab Enterprise, but you do not have permission "
            "to create this Cloud Storage bucket. Set GCS_BUCKET above to an "
            "existing bucket where you have object read/write access."
        ) from exc

print(f"Durable checkpoint root: gs://{GCS_BUCKET}/{GCS_PREFIX}/")


## 3. Select run type and restore any existing checkpoint

Use `test` first, then `pilot`, then `production`. Each run type has a separate Cloud Storage prefix.


In [ ]:
RUN_TYPE = "test"   # test | pilot | production

LOCAL_OUTPUT_DIR = RUNTIME_ROOT / "outputs" / RUN_TYPE
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GCS_OUTPUT_PREFIX = f"{GCS_PREFIX}/outputs/{RUN_TYPE}"

RUNTIME_CONFIG = RUNTIME_ROOT / f"chembreak_v11_{RUN_TYPE}.json"
cfg = json.loads(CONFIG_SOURCE.read_text(encoding="utf-8"))
cfg["project_id"] = PROJECT_ID
cfg["run_type"] = RUN_TYPE
RUNTIME_CONFIG.write_text(json.dumps(cfg, indent=2), encoding="utf-8")

_uploaded_fingerprints = {}

def _stable_bytes(path: Path):
    try:
        s1 = path.stat()
        data = path.read_bytes()
        s2 = path.stat()
    except FileNotFoundError:
        return None, None

    fp1 = (s1.st_size, s1.st_mtime_ns)
    fp2 = (s2.st_size, s2.st_mtime_ns)
    if fp1 != fp2:
        return None, None
    return data, fp2

def sync_to_gcs(verbose=True):
    uploaded = 0
    for path in LOCAL_OUTPUT_DIR.rglob("*"):
        if not path.is_file():
            continue

        data, fp = _stable_bytes(path)
        if data is None:
            continue

        rel = path.relative_to(LOCAL_OUTPUT_DIR).as_posix()
        if _uploaded_fingerprints.get(rel) == fp:
            continue

        blob = bucket.blob(f"{GCS_OUTPUT_PREFIX}/{rel}")
        blob.upload_from_string(data)
        _uploaded_fingerprints[rel] = fp
        uploaded += 1

    if verbose:
        print(
            f"GCS sync complete: {uploaded} changed file(s) -> "
            f"gs://{GCS_BUCKET}/{GCS_OUTPUT_PREFIX}/"
        )
    return uploaded

def restore_from_gcs():
    prefix = f"{GCS_OUTPUT_PREFIX}/"
    restored = 0
    for blob in storage_client.list_blobs(GCS_BUCKET, prefix=prefix):
        rel = blob.name[len(prefix):]
        if not rel or rel.endswith("/"):
            continue

        target = LOCAL_OUTPUT_DIR / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        blob.download_to_filename(str(target))
        restored += 1

    print(
        f"Checkpoint restore: {restored} file(s) <- "
        f"gs://{GCS_BUCKET}/{GCS_OUTPUT_PREFIX}/"
    )
    return restored

restore_from_gcs()

print("Run type:", RUN_TYPE)
print("Local working output:", LOCAL_OUTPUT_DIR)
print("Durable output:", f"gs://{GCS_BUCKET}/{GCS_OUTPUT_PREFIX}/")


## 4. Stage runner with live output and background checkpoint mirroring


In [ ]:
def run_stage(stage):
    command = [
        sys.executable, "-u", str(PIPELINE),
        "--stage", stage,
        "--project-dir", str(PROJECT_DIR),
        "--config", str(RUNTIME_CONFIG),
        "--output-dir", str(LOCAL_OUTPUT_DIR),
    ]

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
    env["GOOGLE_CLOUD_QUOTA_PROJECT"] = PROJECT_ID

    stop_sync = threading.Event()

    def _mirror_loop():
        while not stop_sync.wait(GCS_SYNC_SECONDS):
            try:
                changed = sync_to_gcs(verbose=False)
                if changed:
                    print(
                        f"\n[V11 GCS CHECKPOINT] mirrored {changed} changed file(s)",
                        flush=True,
                    )
            except Exception as exc:
                print(
                    f"\n[V11 GCS CHECKPOINT WARNING] {type(exc).__name__}: {exc}",
                    flush=True,
                )

    mirror_thread = threading.Thread(
        target=_mirror_loop,
        name=f"v11-gcs-sync-{stage}",
        daemon=True,
    )

    started = time.time()
    print(f"\n===== V11 {stage.upper()} START =====", flush=True)
    print("Durable checkpoint:", f"gs://{GCS_BUCKET}/{GCS_OUTPUT_PREFIX}/", flush=True)

    mirror_thread.start()

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    assert process.stdout is not None

    try:
        for line in process.stdout:
            print(line, end="", flush=True)
    finally:
        process.stdout.close()
        return_code = process.wait()
        stop_sync.set()
        mirror_thread.join(timeout=10)

        # Always make one final durable sync, including after a stage error.
        try:
            sync_to_gcs(verbose=True)
        except Exception as exc:
            print(
                f"[V11 FINAL GCS SYNC WARNING] {type(exc).__name__}: {exc}",
                flush=True,
            )

    elapsed = time.time() - started

    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

    print(
        f"===== V11 {stage.upper()} DONE | elapsed {elapsed/60:.1f} min =====\n",
        flush=True,
    )

print("Enterprise stage runner ready.")


## 5. Preflight


In [ ]:
run_stage("preflight")


## 6. Bootstrap source data


In [ ]:
run_stage("bootstrap")


## 7. Create V11 assignment plan


In [ ]:
run_stage("plan")


## 8. Generate the candidate pool


In [ ]:
run_stage("generate")


## 9. Deterministic validation


In [ ]:
run_stage("validate")


## 10. Repair invalid candidates


In [ ]:
run_stage("repair")


## 11. Pre-judge recovery

If three candidates are valid, all three proceed.
If two are valid, both proceed.
If one is valid, V11 tries targeted pre-judge refill before falling back to dual single-candidate qualification.
If none are valid, the later full refill stage handles the assignment.


In [ ]:
run_stage("prejudge_refill")


## 12. Two independent judges, concurrent


In [ ]:
run_stage("judge")


## 13. Blind adjudication for genuine disagreement


In [ ]:
run_stage("adjudicate")


## 14. Full refill cycles

This loop is safe to rerun. V11 checkpoints completed work and only acts on unresolved assignments.


In [ ]:
for cycle in range(int(cfg["recovery"]["max_full_refill_cycles"])):
    print(f"\n===== V11 FULL REFILL CYCLE {cycle + 1} =====")
    run_stage("refill")
    run_stage("prejudge_refill")
    run_stage("judge")
    run_stage("adjudicate")


## 15. Finalize the task bank


In [ ]:
run_stage("finalize")


## 16. Final status and durable checkpoint


In [ ]:
run_stage("status")
sync_to_gcs(verbose=True)

print("\nV11 Enterprise run complete.")
print("Durable results:", f"gs://{GCS_BUCKET}/{GCS_OUTPUT_PREFIX}/")


## Resume after a runtime restart or deletion

Reconnect or create a new Colab Enterprise runtime and rerun Sections 1-4.

Section 3 restores the existing checkpoint from Cloud Storage. Then rerun the stage that was interrupted. V11's checkpoint files and run-compatibility guard determine what is already complete, so completed work is not intentionally regenerated.
